In [3]:
from langgraph.graph import StateGraph,START, END , MessagesState
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict
load_dotenv()

True

In [2]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [4]:
def chat(state : MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages" : response}

In [10]:
graph = StateGraph(MessagesState)
graph.add_node("Chat_Node",chat)
graph.add_edge(START, "Chat_Node")
graph.add_edge("Chat_Node", END)
workflow = graph.compile()

In [12]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is the capital of France?"}]})

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='7e8253aa-fd6a-41fe-9a82-8ae546d53edb'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-0f1724ea-c530-483b-afbc-53790f01b97e-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}})]}

In [13]:
workflow.invoke({"messages" : [{"role": "user", "content": "My name is John"}]})

{'messages': [HumanMessage(content='My name is John', additional_kwargs={}, response_metadata={}, id='b0a6a119-3752-4301-83bb-d7bc1bb4bf40'),
  AIMessage(content='Hi John!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-9c2bf331-eaf5-461e-9057-3063d75e7671-0', usage_metadata={'input_tokens': 5, 'output_tokens': 3, 'total_tokens': 253, 'input_token_details': {'cache_read': 0}})]}

In [14]:
workflow.invoke({"messages" : [{"role": "user", "content": "What is my name?"}]})

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='6e700d69-40aa-4937-bab1-c28e0d0c368d'),
  AIMessage(content="As an AI, I don't have access to personal information about you, including your name. I don't know who you are.\n\nIf you'd like to tell me your name for our current conversation, I can refer to you by it!", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-388a4269-6904-48dc-8fcd-47c2e9330394-0', usage_metadata={'input_tokens': 6, 'output_tokens': 53, 'total_tokens': 511, 'input_token_details': {'cache_read': 0}})]}

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [7]:
def chat2(state : MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages" : response}

In [8]:
graph2= StateGraph(MessagesState)
graph2.add_node("Chat_Node",chat)
graph2.add_edge(START, "Chat_Node")
graph2.add_edge("Chat_Node", END)
workflow2 = graph2.compile(checkpointer=checkpointer)

In [9]:
config = {"configurable":{"thread_id": "thread_1"}}

In [15]:
workflow2.invoke({"messages" : [{"role": "user", "content": "my name is john.What is the capital of France?"}]},config=config)

{'messages': [HumanMessage(content='my name is john.What is the capital of France?', additional_kwargs={}, response_metadata={}, id='66e9ffd5-6a3e-4507-83be-5ea7a27c35c5'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-277d241b-c011-4219-9f56-63d9f5fec043-0', usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}})]}

In [16]:
workflow2.invoke({"messages" : [{"role": "user", "content": "What is my name?"}]},config=config)

{'messages': [HumanMessage(content='my name is john.What is the capital of France?', additional_kwargs={}, response_metadata={}, id='66e9ffd5-6a3e-4507-83be-5ea7a27c35c5'),
  AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-277d241b-c011-4219-9f56-63d9f5fec043-0', usage_metadata={'input_tokens': 13, 'output_tokens': 8, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}}),
  HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='157fecc7-c3ea-4a0e-9182-f15d62d30359'),
  AIMessage(content='Your name is **John**.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-4af35f6c-624e-4c18-84fc-a421a91182a6-0', usage_metadata={'input_tokens': 28, 'output_tokens': 6, 'total_tokens':

In [20]:
snap = workflow2.get_state(config=config)
vals = snap.values
for m in vals.get("messages",[]):
    print("-" ,type(m).__name__,":",m.content)

- HumanMessage : my name is john.What is the capital of France?
- AIMessage : The capital of France is **Paris**.
- HumanMessage : What is my name?
- AIMessage : Your name is **John**.
